# **Step 1: Dataset Preparation & Splitting**
### **Helmet Detection (With Helmet vs Without Helmet)**

This notebook handles the data preparation and conversion pipeline:
1. **Load Raw Dataset**: Parse Pascal VOC XML annotations and images from `data/`.
2. **Data Preprocessing**: Split into **Train (80%)**, **Validation (10%)**, and **Test (10%)**.
3. **YOLO Conversion**: Export bounding box annotations to standard YOLO format (`HelmetDataset/`).
4. **Configuration**: Generate and verify `HelmetDataset/data.yaml` with all required YOLO keys (`path`, `train`, `val`, `test`).
5. **Data Exploration**: Check file split counts and visualize random training samples.


## **1. Import Modules & Setup Environment**

In [ ]:
import os
import shutil
import random
import glob
import cv2
import yaml
import matplotlib.pyplot as plt
import supervision as sv

print(f"Supervision version: {sv.__version__}")


## **2. Define Dataset Paths & Setup Directory**

In [ ]:
BASE_DIR = os.getcwd()

if os.path.exists('/kaggle/input/helmet-detection/images'):
    IMG_DIR = '/kaggle/input/helmet-detection/images'
    ANN_DIR = '/kaggle/input/helmet-detection/annotations'
    FINAL_DIR = '/kaggle/working/HelmetDataset'
elif os.path.exists(os.path.join(BASE_DIR, 'data', 'images')):
    IMG_DIR = os.path.abspath(os.path.join(BASE_DIR, 'data', 'images'))
    ANN_DIR = os.path.abspath(os.path.join(BASE_DIR, 'data', 'annotations'))
    FINAL_DIR = os.path.abspath(os.path.join(BASE_DIR, 'HelmetDataset'))
else:
    IMG_DIR = os.path.abspath('data/images')
    ANN_DIR = os.path.abspath('data/annotations')
    FINAL_DIR = os.path.abspath('HelmetDataset')

print("Image Directory:      ", IMG_DIR, "(exists:", os.path.exists(IMG_DIR), ")")
print("Annotation Directory: ", ANN_DIR, "(exists:", os.path.exists(ANN_DIR), ")")
print("Target YOLO Directory:", FINAL_DIR)

os.makedirs(FINAL_DIR, exist_ok=True)


## **3. Load Pascal VOC Dataset using Supervision**

In [ ]:
# Load Pascal VOC XML annotations using supervision
dataset = sv.DetectionDataset.from_pascal_voc(
    images_directory_path=IMG_DIR,
    annotations_directory_path=ANN_DIR
)

print(f"Total Images Loaded: {len(dataset)}")
print(f"Classes Detected   : {dataset.classes}")


## **4. Data Preprocessing & Splitting (80% Train, 10% Valid, 10% Test)**

In [ ]:
# Split Dataset (80% train, 10% valid, 10% test)
train_dataset, remaining_dataset = dataset.split(split_ratio=0.8)
valid_dataset, test_dataset = remaining_dataset.split(split_ratio=0.5)

print(f"Train Samples: {len(train_dataset)}")
print(f"Valid Samples: {len(valid_dataset)}")
print(f"Test Samples : {len(test_dataset)}")


## **5. Export to YOLO Format**

In [ ]:
# Export Train, Valid, and Test splits into YOLO format
train_dataset.as_yolo(
    images_directory_path=os.path.join(FINAL_DIR, "train/images"),
    annotations_directory_path=os.path.join(FINAL_DIR, "train/labels")
)

valid_dataset.as_yolo(
    images_directory_path=os.path.join(FINAL_DIR, "valid/images"),
    annotations_directory_path=os.path.join(FINAL_DIR, "valid/labels")
)

test_dataset.as_yolo(
    images_directory_path=os.path.join(FINAL_DIR, "test/images"),
    annotations_directory_path=os.path.join(FINAL_DIR, "test/labels")
)

print("✅ YOLO export completed successfully!")


## **6. Configure & Validate data.yaml for YOLOv8**

In [ ]:
# Fix and verify data.yaml with all required keys for Ultralytics YOLO
DATA_YAML_PATH = os.path.join(FINAL_DIR, 'data.yaml')

data = {
    'path': FINAL_DIR.replace('\\', '/'),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(dataset.classes),
    'names': dataset.classes
}

with open(DATA_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, sort_keys=False)

print(f"✅ data.yaml verified and created at: {DATA_YAML_PATH}\n")
with open(DATA_YAML_PATH, 'r') as f:
    print(f.read())


## **7. Verify Dataset Split File Counts**

In [ ]:
# Check counts of images and labels across splits
for split in ['train', 'valid', 'test']:
    img_count = len(glob.glob(os.path.join(FINAL_DIR, split, 'images', '*.*')))
    lbl_count = len(glob.glob(os.path.join(FINAL_DIR, split, 'labels', '*.txt')))
    print(f"{split.upper():5} -> Images: {img_count:3} | Labels: {lbl_count:3}")


## **8. Visualize Random Training Images**

In [ ]:
# Visualize Random Training Images
TRAIN_IMG_DIR = os.path.join(FINAL_DIR, 'train', 'images')
train_images = os.listdir(TRAIN_IMG_DIR)
sample_images = random.sample(train_images, min(9, len(train_images)))

plt.figure(figsize=(12, 12))
for i, image_name in enumerate(sample_images):
    image_path = os.path.join(TRAIN_IMG_DIR, image_name)
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(image_name, fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()


## **Next Step**
The dataset is now formatted in YOLO structure under `HelmetDataset/`.
👉 Open **`2_train_model.ipynb`** to train the YOLOv8 object detection model!
